In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.path.abspath(".."))

from src.data import get_xy
from src.models import evaluate_model, print_results, print_test_metrics, get_permutation_importance, print_coefficients, bootstrap_logistic_inference
from src import visalization as viz

In [ ]:
df_train = pd.read_excel("training_barcelona_shots.xlsx", header=0)
df_test = pd.read_excel("testing_barcelona_shots.xlsx", header=0)

shot_features = [
    "distance_d",
    "angle_d",
    "free_kick_flag",
    "penalty_flag",
    "technique_b",
    "n_def_1_5",
    "n_def_3_0",
    "dist_nearest_def",
    "gk_dist_to_shooter"
]

X_train, y_train = get_xy(df_train, shot_features)
X_test, y_test = get_xy(df_test, shot_features)

In [ ]:
model_xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 1, 5]
}

In [ ]:
grid_xgb, results_xgb = evaluate_model(model_xgb, param_grid_xgb, X_train, y_train)
print_results(results_xgb)

In [ ]:
final_model_xgb = XGBClassifier(**grid_xgb.best_params_, eval_metric="logloss", random_state=42)
final_model_xgb.fit(X_train, y_train)

y_pred_proba_xgb = final_model_xgb.predict_proba(X_test)[:, 1]

In [ ]:
df_test["predicted_xg_xgb"] = y_pred_proba_xgb

statsbomb_xg = df_test["statsbomb_xg"]
pred_xg_xgb = df_test["predicted_xg_xgb"]

total_pred_xg_xgb = pred_xg_xgb.sum()
total_statsbomb_xg = statsbomb_xg.sum()
total_goals = y_test.sum()

print(f"Total Predicted xG: {total_pred_xg_xgb:.2f}")
print(f"Total StatsBomb xG: {total_statsbomb_xg:.2f}")
print(f"Actual Goals: {total_goals}")

In [ ]:
print_test_metrics(y_test, pred_xg_xgb, statsbomb_xg)
correlation_xgb = np.corrcoef(statsbomb_xg, pred_xg_xgb)[0, 1]
mae_xgb = np.mean(np.abs(statsbomb_xg - pred_xg_xgb))
print(f"\nCorrelation (Model xG vs StatsBomb xG): {correlation_xgb:.3f}")
print(f"Mean Absolute Error: {mae_xgb:.3f}")

In [ ]:
viz.plot_correlation(statsbomb_xg, pred_xg_xgb)

In [ ]:
viz.plot_calibration(y_test, pred_xg_xgb, statsbomb_xg)

In [ ]:
week_xg_xgb = df_test.groupby('week').agg({
    'predicted_xg_xgb': 'sum',
    'statsbomb_xg': 'sum',
    'goal/no goal': 'sum'
}).reset_index()

viz.plot_weekly_xg(week_xg_xgb, 'predicted_xg_xgb')

In [ ]:
viz.plot_roc_curve(y_test, pred_xg_xgb, statsbomb_xg)

In [ ]:
importances_df_xgb = get_permutation_importance(final_model_xgb, X_test.columns, X_test, y_test)
viz.plot_feature_importances(importances_df_xgb, "XGBoost Classifier")

In [ ]:
scaler_lr = StandardScaler()

X_train_scaled_lr = scaler_lr.fit_transform(X_train)
X_test_scaled_lr = scaler_lr.transform(X_test)

In [ ]:
C_values = np.logspace(-6, 3, 50)

coefs = []

for C in C_values:
    lr = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000,
        C=C,
        random_state=42
    )
    lr.fit(X_train_scaled_lr, y_train)
    coefs.append(lr.coef_[0])

coefs = np.array(coefs)
viz.plot_l1_paths(C_values, coefs, X_train.columns)

In [ ]:
model_lr = LogisticRegression(
    max_iter=1000, 
    random_state=42
)

param_grid_lr = {
    "penalty": ["l1"],
    "C": [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1],
    "solver": ["liblinear"]
}

In [ ]:
grid_lr, results_lr = evaluate_model(model_lr, param_grid_lr, X_train_scaled_lr, y_train)
print_results(results_lr)

In [ ]:
best_lr = grid_lr.best_estimator_
coef = best_lr.coef_[0]
print_coefficients(X_train.columns, coef)

In [ ]:
final_log_reg = LogisticRegression(**grid_lr.best_params_, max_iter=1000, random_state=42)
final_log_reg.fit(X_train_scaled_lr, y_train)

y_pred_proba_lr = final_log_reg.predict_proba(X_test_scaled_lr)[:, 1]

In [ ]:
shot_features_after_l1_regularization = [
    "angle_d",
    "free_kick_flag",
    "n_def_3_0",
    "dist_nearest_def",
    "gk_dist_to_shooter"
]

X_train_l2, y_train_l2 = get_xy(df_train, shot_features_after_l1_regularization)
X_test_l2, y_test_l2 = get_xy(df_test, shot_features_after_l1_regularization)

X_train_scaled_lr_l2 = scaler_lr.fit_transform(X_train_l2)
X_test_scaled_lr_l2 = scaler_lr.transform(X_test_l2)

In [ ]:
model_lr_l2 = LogisticRegression(
    max_iter=1000, 
    random_state=42
)

param_grid_lr_l2 = {
    "penalty": ["l2"],
    "C": [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["liblinear"]
}

In [ ]:
grid_lr_l2, results_lr_l2 = evaluate_model(model_lr_l2, param_grid_lr_l2, X_train_scaled_lr_l2, y_train_l2)
print_results(results_lr_l2)

In [ ]:
best_lr_l2 = grid_lr_l2.best_estimator_
coef_l2 = best_lr_l2.coef_[0]
print_coefficients(X_train_l2.columns, coef_l2)

In [ ]:
final_log_reg_l2 = LogisticRegression(**grid_lr_l2.best_params_, max_iter=1000, random_state=42)
final_log_reg_l2.fit(X_train_scaled_lr_l2, y_train_l2)

y_pred_proba_lr_l2 = final_log_reg_l2.predict_proba(X_test_scaled_lr_l2)[:, 1]

In [ ]:
df_test["predicted_xg_lr"] = y_pred_proba_lr_l2

statsbomb_xg = df_test["statsbomb_xg"]
pred_xg_lr = df_test["predicted_xg_lr"]

total_pred_xg_lr = pred_xg_lr.sum()
total_statsbomb_xg = statsbomb_xg.sum()
total_goals = y_test.sum()

print(f"Total Predicted xG: {total_pred_xg_lr:.2f}")
print(f"Total StatsBomb xG: {total_statsbomb_xg:.2f}")
print(f"Actual Goals: {total_goals}")

In [ ]:
print_test_metrics(y_test, pred_xg_lr, statsbomb_xg)
correlation_lr = np.corrcoef(statsbomb_xg, pred_xg_lr)[0, 1]
mae_lr = np.mean(np.abs(statsbomb_xg - pred_xg_lr))
print(f"\nCorrelation (Model xG vs StatsBomb xG): {correlation_lr:.3f}")
print(f"Mean Absolute Error: {mae_lr:.3f}")

In [ ]:
viz.plot_correlation(statsbomb_xg, pred_xg_lr)

In [ ]:
viz.plot_calibration(y_test, pred_xg_lr, statsbomb_xg)

In [ ]:
week_xg_lr = df_test.groupby('week').agg({
    'predicted_xg_lr': 'sum',
    'statsbomb_xg': 'sum',
    'goal/no goal': 'sum'
}).reset_index()

viz.plot_weekly_xg(week_xg_lr, 'predicted_xg_lr')

In [ ]:
viz.plot_roc_curve(y_test, pred_xg_lr, statsbomb_xg)

In [ ]:
importances_df_lr = get_permutation_importance(final_log_reg_l2, X_test_l2.columns, X_test_scaled_lr_l2, y_test)
viz.plot_feature_importances(importances_df_lr, "Logistic regression")

In [ ]:
results = bootstrap_logistic_inference(final_log_reg_l2, X_train_scaled_lr_l2, y_train, X_train_l2.columns)

print("\nL2-penalized Logistic Regression — Bootstrap Inference (2000 resamples)\n")
print(results.to_string(index=False))

In [ ]:
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
}

In [ ]:
grid_rf, results_rf = evaluate_model(model_rf, param_grid_rf, X_train, y_train)
print_results(results_rf)

In [ ]:
final_model_rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42, n_jobs=-1)
final_model_rf.fit(X_train, y_train)

y_pred_proba_rf = final_model_rf.predict_proba(X_test)[:, 1]

In [ ]:
df_test["predicted_xg_rf"] = y_pred_proba_rf
pred_xg_rf = df_test["predicted_xg_rf"]
total_pred_xg_rf = pred_xg_rf.sum()

print(f"Total Predicted xG: {total_pred_xg_rf:.2f}")
print(f"Total StatsBomb xG: {total_statsbomb_xg:.2f}")
print(f"Actual Goals: {total_goals}")

In [ ]:
print_test_metrics(y_test, pred_xg_rf, statsbomb_xg)
correlation_rf = np.corrcoef(statsbomb_xg, pred_xg_rf)[0, 1]
mae_rf = np.mean(np.abs(statsbomb_xg - pred_xg_rf))
print(f"\nCorrelation (Model xG vs StatsBomb xG): {correlation_rf:.3f}")
print(f"Mean Absolute Error: {mae_rf:.3f}")

In [ ]:
viz.plot_correlation(statsbomb_xg, pred_xg_rf)

In [ ]:
viz.plot_calibration(y_test, pred_xg_rf, statsbomb_xg)

In [ ]:
week_xg_rf = df_test.groupby('week').agg({
    'predicted_xg_rf': 'sum',
    'statsbomb_xg': 'sum',
    'goal/no goal': 'sum'
}).reset_index()

viz.plot_weekly_xg(week_xg_rf, 'predicted_xg_rf')

In [ ]:
viz.plot_roc_curve(y_test, pred_xg_rf, statsbomb_xg)

In [ ]:
importances_df_rf = get_permutation_importance(final_model_rf, X_test.columns, X_test, y_test)
viz.plot_feature_importances(importances_df_rf, "Random Forest Classifier")